# 第 05 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-05-04_chapter_test"></a>


## 5.4 章节实践

参考前两节补写设备调用，完成下面两组输入的计算和 NumPy 校验。

### 一、基础题

1. `acl.op.set_model_dir` 的作用是：A. 加载整图模型；B. 注册单算子 OM 目录；C. 创建 Stream。
2. H2D 与 D2H 分别表示什么方向？为什么异步提交算子后要先同步再回读？
3. 本实验的 `SparseTensorDenseMatMul` 使用 AI Core 还是 AI CPU/tf_kernel？
4. 为什么 `acl` 可以导入、OM 文件存在，还不能说明计算结果正确？


In [ ]:
import json
import subprocess
import time
from pathlib import Path
import acl
%run src/acl_helpers.py

DEVICE_ID = 0
REPEAT = 10
SOC_VERSION = acl.get_soc_name()


### 二、GEMM

使用 `M=32、N=64、K=128`，通过 `acl.blas.gemm_ex` 计算。与下面的 NumPy 结果比较，记录最大误差和平均执行时间。


In [ ]:
# 题目给定的输入参数；在后面补写设备调用。
import numpy as np

M, N, K = 32, 64, 128
gemm_a = ((np.arange(M)[:, None] + np.arange(K)[None, :]) % 7).astype(np.float16)
gemm_b = ((np.arange(K)[:, None] * 2 + np.arange(N)[None, :]) % 9).astype(np.float16)
gemm_golden = (gemm_a.astype(np.float32) @ gemm_b.astype(np.float32)).astype(np.float16)
assert gemm_golden.shape == (M, N)
print("GEMM practice input:", gemm_a.shape, gemm_b.shape, gemm_golden.shape)


### 三、COO SpMV

使用 `M=64、N=128、NNZ=100`，通过 `acl.op.execute_v2("SparseTensorDenseMatMul", ...)` 计算。与下面按 COO 累加的 NumPy 结果比较，记录最大误差和平均执行时间。该算子在本实验中使用 AI CPU/tf_kernel。


In [ ]:
# 题目给定的 COO 输入；在后面补写设备调用。
import numpy as np

SM, SN, NNZ = 64, 128, 100
linear = np.arange(NNZ, dtype=np.int64) * (SM * SN // NNZ)
spmv_indices = np.stack([linear // SN, linear % SN], axis=1)
spmv_values = ((np.arange(NNZ) * 5 + 1) % 11 + 1).astype(np.float32)
spmv_x = ((np.arange(SN) * 3 + 1) % 13).astype(np.float32)

assert spmv_indices.shape == (NNZ, 2)
assert np.unique(spmv_indices, axis=0).shape[0] == NNZ
assert np.all((0 <= spmv_indices[:, 0]) & (spmv_indices[:, 0] < SM))
assert np.all((0 <= spmv_indices[:, 1]) & (spmv_indices[:, 1] < SN))
spmv_golden = np.zeros(SM, dtype=np.float32)
for (row, col), value in zip(spmv_indices, spmv_values):
    spmv_golden[row] += value * spmv_x[col]
print("SpMV practice input:", spmv_indices.shape, spmv_values.shape, spmv_x.shape)


### 四、整理成函数

将设备调用分别写成 `run_gemm()` 和 `run_spmv()`，使用本题输入更新单算子描述并编译。资源管理可沿用 `AclSession`。

两个函数都要同步后回读，并用 `np.testing.assert_allclose` 检查结果：GEMM 使用 `atol=0.02`，SpMV 使用 `atol=1e-3`，两者均设 `rtol=0`。函数返回输出数组，打印平均执行时间和最大误差。


In [ ]:
# 学生补写：run_gemm() 和 run_spmv()，返回设备计算的输出数组。
# 完成后取消下面四行的注释，检查结果。
# gemm_output = run_gemm()
# np.testing.assert_allclose(gemm_output, gemm_golden, atol=0.02, rtol=0)
# spmv_output = run_spmv()
# np.testing.assert_allclose(spmv_output.reshape(-1), spmv_golden, atol=1e-3, rtol=0)


### 完成标准

两种算子都完成设备调用、同步、回读和 NumPy 校验，并能解释结果形状、执行位置和计时范围。


完成本节练习后查看：[参考答案](answer/05.04_chapter_test_answer.md)